In [2]:
import sys
sys.path.append('/n/home06/drooryck/codeswitching-llms')
from pathlib import Path
from july_aug_exp.src.metrics import Metrics
from july_aug_exp.src.dataset_manager import DatasetManager
from july_aug_exp.src.model_config import ModelConfig, SlurmConfig
from july_aug_exp.src.experiment import Experiment
from july_aug_exp.src.plotting import BilingualPlotter

In [ ]:
# mange, voit, aime, cherche, aide, suit, frappe, porte, embrasse, chasse, obverse, soigne, poursuit, attaque, sauve, reconforte, repousse, regarde, ignore

# eet, ziet, mag, zoekt, helpt, volgt, slaat, draagt, zoent, jaagt, observeert, verzorgt, achtervolgt, valt, redt, troost, duwt, bekijkt negeert

In [6]:
# Initialize with minimal config
repo_root = Path("/n/home06/drooryck/codeswitching-llms/july_aug_exp/")
config = ModelConfig.load(repo_root / "configs" / "default_model.json")  # or create a basic one
lexicon_path = "/n/home06/drooryck/codeswitching-llms/july_aug_exp/data/lexicon_sep22.json"
data_manager = DatasetManager(str(repo_root / "data"), config=config, lexicon_path=lexicon_path)

# Generate train/test splits
# This will:
# 1. Create balanced splits ensuring (lang, subj, verb) combinations don't overlap
# 2. Save to new_train.csv and new_test.csv in the data directory
# 3. Return the dataframes for immediate use
train_df, test_df = data_manager.make_and_save_testing_and_training_data(
    test_size=0.2,  # 20% test set
    random_seed=42  # for reproducibility
)

# You can then examine the data:
print("Training set size:", len(train_df))
print("Test set size:", len(test_df))
print("\nTraining set columns:", train_df.columns.tolist())
print("\nSample rows:")
print(train_df.head())

# Check language distribution
print("\nLanguage distribution in train:")
print(train_df['lang'].value_counts())
print("\nLanguage distribution in test:")
print(test_df['lang'].value_counts())

test_df

Training set size: 262936
Test set size: 65688

Training set columns: ['input', 'target', 'lang', 'plural', 'subj', 'obj', 'verb']

Sample rows:
                          input                          target lang  plural  \
0        le chien mange le loup        le chien a mangé le loup   fr   False   
1  les chiens mangent les loups  les chiens ont mangé les loups   fr    True   
2         le chien voit le loup           le chien a vu le loup   fr   False   
3   les chiens voient les loups     les chiens ont vu les loups   fr    True   
4      le chien cherche le loup      le chien a cherché le loup   fr   False   

    subj   obj     verb  
0  chien  loup    mange  
1  chien  loup    mange  
2  chien  loup     voit  
3  chien  loup     voit  
4  chien  loup  cherche  

Language distribution in train:
lang
fr    133492
nl    129444
Name: count, dtype: int64

Language distribution in test:
lang
nl    34868
fr    30820
Name: count, dtype: int64


,input,target,lang,plural,subj,obj,verb
0,le chien aime le loup,le chien a aimé le loup,fr,False,chien,loup,aime
1,les chiens aiment les loups,les chiens ont aimé les loups,fr,True,chien,loup,aime
2,le chien frappe le loup,le chien a frappé le loup,fr,False,chien,loup,frappe
3,les chiens frappent les loups,les chiens ont frappé les loups,fr,True,chien,loup,frappe
4,le chien attaque le loup,le chien a attaqué le loup,fr,False,chien,loup,attaque
...,...,...,...,...,...,...,...
65683,de spinnen redden de hagedissen,de spinnen hebben de hagedissen gered,nl,True,spin,hagedis,redt
65684,de spin troost de hagedis,de spin heeft de hagedis getroost,nl,False,spin,hagedis,troost
65685,de spinnen troosten de hagedissen,de spinnen hebben de hagedissen getroost,nl,True,spin,hagedis,troost
65686,de spin geeft de hagedis,de spin heeft de hagedis gegeven,nl,False,spin,hagedis,geeft


## how many tokens we should train on 

In [5]:
import pandas as pd
import json

# config and lexicon path alr exist

# Initialize DatasetManager and build tokenizer
data_manager = DatasetManager("/n/home06/drooryck/codeswitching-llms/july_aug_exp/data", config, lexicon_path)
tokenizer = data_manager.build_tokenizer()

# Load training data
train_df = pd.read_csv("/n/home06/drooryck/codeswitching-llms/july_aug_exp//data/train.csv")

# Count tokens in each example (input + target)
total_tokens = 0
for _, row in train_df.iterrows():
    # Format matches your training format: <sos> input <sep> target <eos>
    full_text = f"<sos> {row['input']} <sep> {row['target']} <eos>"
    tokens = tokenizer(full_text)['input_ids']
    total_tokens += len(tokens)

print(f"Total examples: {len(train_df)}")
print(f"Total tokens in training: {total_tokens}")
print(f"Average tokens per example: {total_tokens/len(train_df):.1f}")

# Chinchilla analysis
params = 13_218_304  # from our previous calculation
chinchilla_recommended_tokens = params * 20  # Chinchilla recommends 20x tokens to params

print(f"\nChinchilla analysis:")
print(f"Our parameters: {params:,}")
print(f"Our training tokens: {total_tokens:,}")
print(f"Chinchilla recommended tokens: {chinchilla_recommended_tokens:,}")
print(f"We are training with {total_tokens/chinchilla_recommended_tokens:.1%} of recommended tokens")

# Calculate effective training tokens over full training
batch_size = 16
max_steps = 20000
effective_tokens = batch_size * max_steps * (total_tokens/len(train_df))  # tokens seen during training

print(f"\nEffective training:")
print(f"Tokens seen during full training: {effective_tokens:,.0f}")
print(f"Each token seen average of {effective_tokens/total_tokens:.1f} times")

Total examples: 262936
Total tokens in training: 3681104
Average tokens per example: 14.0

Chinchilla analysis:
Our parameters: 13,218,304
Our training tokens: 3,681,104
Chinchilla recommended tokens: 264,366,080
We are training with 1.4% of recommended tokens

Effective training:
Tokens seen during full training: 4,480,000
Each token seen average of 1.2 times


## experiment

In [ ]:
repo_root = Path("/n/home06/drooryck/codeswitching-llms/july_aug_exp/")

output_dir = repo_root / "results" / "oct21"
output_dir.mkdir(parents=True, exist_ok=True)

config = ModelConfig.load(repo_root / "configs" / "default_model.json")
slurm_config = SlurmConfig.load(repo_root / "configs" / "slurm_default.json")
slurm_config.output_pattern = str(output_dir / "logs" / "slurm_%A_%a.out")
slurm_config.error_pattern = str(output_dir / "logs" / "slurm_%A_%a.err")
slurm_config.job_name = "oct21exp" 

data_manager = DatasetManager("data", config, lexicon_path = "/n/home06/drooryck/codeswitching-llms/july_aug_exp/data/lexicon_sep22.json")
metrics = Metrics("/n/home06/drooryck/codeswitching-llms/july_aug_exp/data/lexicon_sep22.json")
experiment = Experiment(config, data_manager, metrics, output_dir)

config.save(output_dir / "model_config.json")
slurm_config.save(output_dir / "slurm_config.json")

props = [0.000, 0.001, 0.005, 0.010, 0.015, 0.020, 0.025, 0.030, 
0.040, 0.050, 0.075, 0.100, 0.150, 0.200, 0.250, 0.300, 0.400, 0.450
, 0.500, 0.550, 0.600, 0.650, 0.700, 0.750, 0.800, 0.850, 0.900, 0.925, 0.950,
 0.960, 0.970, 0.980, 0.985, 0.990, 0.995, 0.999, 1.000]
runs = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20]
# props = [0.5]
# runs = [42]

job_ids = experiment.submit_to_slurm(
    props=props,
    runs=runs,
    slurm_config=slurm_config,
    eval_prop=0.1
)

2025-09-24 15:41:49,999 |     INFO | Submitting 185 jobs to SLURM...


2025-09-24 15:41:50,277 |     INFO | Submitted job array 36993810


In [4]:
from pathlib import Path
import pandas as pd
import json
from typing import List
from july_aug_exp.src.plotting import BilingualPlotter

def collect_ablation_results(run_dirs: List[Path]) -> pd.DataFrame:
    results = []

    for run_dir in run_dirs:
        # Extract run parameters from directory name
        dir_name = run_dir.name
        if dir_name.startswith("p") and "_run" in dir_name:
            parts = dir_name.split("_run")
            try:
                # Handle decimal proportions by removing 'p' and converting directly to float
                prop = float(parts[0][1:]) / 100.0  # Just convert the string after 'p' to float
                run_id = int(parts[1])
            except ValueError:
                print(f"Couldn't parse proportion or run ID from {dir_name}")
                continue
        else:
            continue

        # Track if we found any metrics for this run
        found_metrics = False

        # Collect metrics for each ablation type
        for ablation_type in ['none', 'subject', 'verb', 'object']:
            metrics_file = run_dir / f"ablation_{ablation_type}_metrics.json"
            if not metrics_file.exists():
                continue

            try:
                with open(metrics_file, 'r') as f:
                    metrics = json.load(f)
                found_metrics = True
            except json.JSONDecodeError:
                print(f"Error reading metrics from {metrics_file}")
                continue

            result = {
                'prop': prop,
                'run_id': run_id,
                'run_dir': str(run_dir),
                'ablation': ablation_type,
                **metrics
            }
            results.append(result)

        if not found_metrics:
            print(f"No metrics found in {run_dir}")

    df = pd.DataFrame(results)
    if df.empty:
        print("Warning: No results collected!")
    else:
        print(f"Collected {len(df)} total results across {df['ablation'].nunique()} ablation types")
    
    return df

# Set up directories
output_dir = Path("/n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep24.3/")
run_dirs = list(output_dir.glob("p*_run*"))

# Collect all results
results_df = collect_ablation_results(run_dirs)

# Create plots for each ablation type
for ablation_type in ['none', 'subject', 'verb', 'object']:
    # Create plots directory for this ablation type
    plots_dir = output_dir / f"plots_{ablation_type}"
    plots_dir.mkdir(exist_ok=True)
    
    type_results = results_df[results_df['ablation'] == ablation_type]
    
    if not type_results.empty:
        print(f"\nCreating plots for {ablation_type} ablation")
        print(f"Found {len(type_results)} runs")
        
        # Create plotter with specific output directory for this ablation type
        plotter = BilingualPlotter(type_results, plots_dir)
        plotter.create_all_plots()
        print(f"Saved plots to {plots_dir}")
    else:
        print(f"No results found for {ablation_type} ablation")

print("\nDone!")

Collected 740 total results across 4 ablation types

Creating plots for none ablation
Found 185 runs
Saved plots to /n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep24.3/plots_none

Creating plots for subject ablation
Found 185 runs
Saved plots to /n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep24.3/plots_subject

Creating plots for verb ablation
Found 185 runs
Saved plots to /n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep24.3/plots_verb

Creating plots for object ablation
Found 185 runs
Saved plots to /n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep24.3/plots_object

Done!


## fix ablation_predictions to be from the test.csv
## update the plots with 5 runs averaged (at least)
## copy the slides from gdrive into a new slides and then fix it.

In [6]:
import pandas as pd

df = pd.read_csv("/n/home06/drooryck/codeswitching-llms/july_aug_sept_exp/results/sep24.3/p99.5_run5/ablation_predictions.csv")

subset = df[df['ablation'] == 'none'].tail(20)
subset

,language,input,gold,prediction,ablation
262672,nl,de spin troost de vlinder,de spin heeft de vlinder getroost,de spin heeft de vlinder getroost,none
262676,nl,de spinnen troosten de vlinders,de spinnen hebben de vlinders getroost,de spinnen hebben de vlinders getroost,none
262680,nl,de spin geeft de vlinder,de spin heeft de vlinder gegeven,de spin heeft de vlinder gegeven,none
262684,nl,de spinnen geven de vlinders,de spinnen hebben de vlinders gegeven,de spinnen hebben de vlinders gegeven,none
262688,nl,de spin zoekt de hagedis,de spin heeft de hagedis gezocht,de spin heeft de hagedis gezocht,none
262692,nl,de spinnen zoeken de hagedissen,de spinnen hebben de hagedissen gezocht,de spinnen hebben de hagedissen gezocht,none
262696,nl,de spin volgt de hagedis,de spin heeft de hagedis gevolgd,de spin heeft de hagedis gevolgd,none
262700,nl,de spinnen volgen de hagedissen,de spinnen hebben de hagedissen gevolgd,de spinnen hebben de hagedissen gevolgd,none
262704,nl,de spin draagt de hagedis,de spin heeft de hagedis gedragen,de spin heeft de hagedis gedragen,none
262708,nl,de spinnen dragen de hagedissen,de spinnen hebben de hagedissen gedragen,de spinnen hebben de hagedissen gedragen,none
